# Week 2: NLP Preprocessing Pipeline
## TruthLens — Fake News Detector

**Goal**: Prepare text data for machine learning by cleaning, tokenizing, and vectorizing news articles with TF-IDF.

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
import sys
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import joblib

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

PROJECT_ROOT = None
for candidate in [Path.cwd()] + list(Path.cwd().parents):
    if (candidate / 'backend').exists() and (candidate / 'data').exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise RuntimeError('Could not locate project root containing backend/ and data/')

sys.path.insert(0, str(PROJECT_ROOT))
data_dir = PROJECT_ROOT / 'data'
model_dir = PROJECT_ROOT / 'backend' / 'models'
model_dir.mkdir(parents=True, exist_ok=True)

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


## 2. Load and Inspect Data

In [2]:
real_path = data_dir / 'True.csv'
fake_path = data_dir / 'Fake.csv'

if not real_path.exists() or not fake_path.exists():
    raise FileNotFoundError('True.csv or Fake.csv not found in data directory')

real_df = pd.read_csv(real_path)
fake_df = pd.read_csv(fake_path)

real_df['label'] = 0
fake_df['label'] = 1

# Combine datasets
df = pd.concat([real_df, fake_df], ignore_index=True)

print(f'Loaded real articles: {len(real_df):,}')
print(f'Loaded fake articles: {len(fake_df):,}')
print(f'Total articles: {len(df):,}')
df.head()

Loaded real articles: 21,417
Loaded fake articles: 23,481
Total articles: 44,898


,title,text,subject,date,label
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017",0
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017",0
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017",0
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017",0
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017",0


## 3. Text Cleaning Functions

In [3]:
def normalize_text(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def clean_text(text: str) -> str:
    text = normalize_text(text)
    tokens = nltk.word_tokenize(text)
    filtered = [lemmatizer.lemmatize(token) for token in tokens if token not in stop_words and len(token) > 1]
    return ' '.join(filtered)

# Test cleaning on sample text
sample = 'Breaking: Trump visits the White House on Monday, and the crowd was enormous!'
print(clean_text(sample))

breaking trump visit white house monday crowd enormous


## 4. Apply Cleaning to the Dataset

In [4]:
df['text_clean'] = df['text'].astype(str).apply(clean_text)
df['title_clean'] = df['title'].astype(str).apply(clean_text)
df['combined_text'] = df['title_clean'] + ' ' + df['text_clean']

print(df[['title', 'text_clean', 'combined_text']].head(3).to_string(index=False))
print()
print('Clean text sample length:', df.loc[0, 'text_clean'][:120])

                                                           title                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        

## 5. Feature Engineering with TF-IDF

In [5]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X = tfidf.fit_transform(df['combined_text'])
y = df['label'].values

print('TF-IDF matrix shape:', X.shape)
print('Sample feature names:', tfidf.get_feature_names_out()[:20])

TF-IDF matrix shape: (44898, 5000)
Sample feature names: ['00' '00 pm' '000' '000 people' '10' '10 000' '10 percent' '10 year'
 '100' '100 000' '11' '12' '120' '13' '14' '15' '15 year' '150' '16' '17']


In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('Training shape:', X_train.shape)
print('Test shape:', X_test.shape)

Training shape: (35918, 5000)
Test shape: (8980, 5000)


## 6. Save Preprocessing Artifacts

In [7]:
joblib.dump(tfidf, model_dir / 'tfidf_vectorizer.joblib')
df.to_csv(data_dir / 'cleaned_news.csv', index=False)
print('Saved TF-IDF vectorizer to', model_dir / 'tfidf_vectorizer.joblib')
print('Saved cleaned dataset to', data_dir / 'cleaned_news.csv')

Saved TF-IDF vectorizer to ..\backend\models\tfidf_vectorizer.joblib
Saved cleaned dataset to ..\data\cleaned_news.csv


## 7. Observations & Next Steps

- Cleaned text now contains lowercased lemmatized tokens with stopwords removed.
- Combined title and article text gives a strong feature source for TF-IDF.
- TF-IDF produces a feature matrix suitable for model training.
- Next: train classical ML models in Week 3 with logistic regression, random forest, and naive Bayes.